# 사전실험 C·D — 주파수 대역 교체(Oracle Test)와 탐지기 내부 Feature Drift

인수인계 문서(실험 C·D 설계)를 그대로 구현하고, 결과 해석에 꼭 필요한 조건 하나(고주파 제거)와 입력 크기 비교(640·1280)만 추가했다.

**공통 조건 (문서 5절)**
- UVG-VCM 무손실 기준 영상과 A-3과 같은 CCTV형 H.264 압축 영상 (`bdata`에 있으면 재사용, 없으면 같은 명령으로 생성)
- 압축 단계 1000·500·250 kbps, YOLO26s, 5프레임 간격, 시퀀스별 결과도 보고
- 프레임 번호·GT·압축 영상 대응 유지, 같은 가중치·평가 코드

**실험 C — 주파수 대역별 탐지 중요도**
- 8×8 블록 DCT(휘도), 대역 DC(0)·Low(1~2)·Mid(3~6)·High(7~14), 경계 사전 고정
- 조건: Baseline / Low·Mid·High 교체 / Reference (+ High_remove: 압축 고주파를 0으로)
- 지표: APs·APm·APl, 원본 대비 회복량(%p)·회복 비율, 객체 단위 작은 객체 회복률(track 단위 bootstrap)
- 산출: `C_ap.csv`, `C_object_recovery.csv`, `C_per_sequence.csv`, 그림
- ※ Oracle Test는 원본 정보를 쓰므로 실제 복원 가능 성능이 아니라 대역 정보의 중요성을 뜻함

**실험 D — Feature Drift**
- YOLO26s 고정, 같은 프레임의 원본·압축(3단계)을 한 배치로 넣어 비교
- 위치: Backbone(stride 4·8·16·32의 마지막 층, 구현에서 자동 탐색) + Neck P3·P4·P5(Detect 입력)
- 지표: 상대 오차, 코사인 유사도, 채널 정규화 MSE, 작은 객체 영역 버전(모든 단계)
- 주입 위치 후보: 프레임별 작은 객체 영역 drift와 작은 객체 미탐률의 순위 상관
- 산출: `D_feature_drift.csv`, `D_summary.csv`, `D_injection_candidates.csv`, 그림
- ※ drift가 가장 큰 층이 최적 주입 위치는 아님. 후보 선정 후 소규모 성능 검증 필요

**실행:** Input에 `uvg-build-dataset`(필수), `uvg-b-data-prep`(있으면), GPU T4, Internet On → `TEST_RUN=True`로 Run All(선택 층이 출력되는지 확인) → `False`로 Save & Run All (약 1~1.5시간)

In [1]:
!pip install -q ultralytics==8.4.165 pycocotools

In [2]:
%%writefile cd_lib.py
"""사전실험 E1(순서) · E2(디블로킹) 공통 라이브러리"""
import json, os, subprocess, contextlib, io
import numpy as np, cv2

W, H = 1280, 720
CCTV_CLASSES = [1, 2, 3, 4, 6, 8]   # person, bicycle, car, motorcycle, bus, truck (COCO 순서 + 1)
BALL_CLASSES = [33]

# ---------------- 디코딩 / 인코딩 ----------------
def read_yuv(video, w, h, keep=None, vf=None):
    """yuv420p 그대로 디코딩. keep=None이면 전체 프레임 리스트, 아니면 {i: (y,u,v)}"""
    cmd = ["ffmpeg", "-v", "error", "-i", video]
    if vf: cmd += ["-vf", vf]
    cmd += ["-f", "rawvideo", "-pix_fmt", "yuv420p", "-"]
    fb = w * h * 3 // 2
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE)
    out = [] if keep is None else {}
    i = 0
    while True:
        b = p.stdout.read(fb)
        if len(b) < fb: break
        if keep is None or i in keep:
            a = np.frombuffer(b, np.uint8)
            f = (a[:w*h].reshape(h, w).copy(), a[w*h:w*h*5//4].reshape(h//2, w//2).copy(),
                 a[w*h*5//4:].reshape(h//2, w//2).copy())
            if keep is None: out.append(f)
            else: out[i] = f
        i += 1
    p.wait()
    if p.returncode: raise RuntimeError(f"decode failed: {video}")
    return out, i

def resize_yuv(f, w2, h2):
    """같은 함수(INTER_AREA)로 두 경로 모두 축소 -> 순서 외의 차이 제거"""
    y, u, v = f
    return (cv2.resize(y, (w2, h2), interpolation=cv2.INTER_AREA),
            cv2.resize(u, (w2//2, h2//2), interpolation=cv2.INTER_AREA),
            cv2.resize(v, (w2//2, h2//2), interpolation=cv2.INTER_AREA))

def yuv_to_bgr(f):
    y, u, v = f; h, w = y.shape
    buf = np.concatenate([y.ravel(), u.ravel(), v.ravel()]).reshape(h * 3 // 2, w)
    return cv2.cvtColor(buf, cv2.COLOR_YUV2BGR_I420)

def encode(frames, w, h, out, mode, val, x264_params=None, fps=30):
    """CCTV형 x264 (veryfast, zerolatency, B프레임 없음, GOP 60). mode: 'kbps'(1-pass CBR형) 또는 'qp'(고정 QP)"""
    cmd = ["ffmpeg", "-y", "-v", "error", "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{w}x{h}",
           "-r", str(fps), "-i", "-", "-c:v", "libx264", "-preset", "veryfast", "-tune", "zerolatency",
           "-profile:v", "high", "-pix_fmt", "yuv420p", "-g", "60", "-keyint_min", "60", "-sc_threshold", "0"]
    if mode == "kbps":
        cmd += ["-b:v", f"{val}k", "-maxrate", f"{val}k", "-bufsize", f"{val}k"]
    else:
        cmd += ["-qp", str(val)]
    if x264_params: cmd += ["-x264-params", x264_params]
    cmd += ["-an", out]
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    for (y, u, v) in frames:
        p.stdin.write(y.tobytes()); p.stdin.write(u.tobytes()); p.stdin.write(v.tobytes())
    p.stdin.close(); p.wait()
    if p.returncode: raise RuntimeError(f"encode failed: {out}")

def kbps_of(p):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=bit_rate", "-of", "csv=p=0", p],
                       capture_output=True, text=True).stdout.strip()
    return round(int(r) / 1000, 1) if r.isdigit() else None

def has_filter(name):
    r = subprocess.run(["ffmpeg", "-hide_banner", "-filters"], capture_output=True, text=True).stdout
    return any(l.split()[1:2] == [name] for l in r.splitlines() if len(l.split()) > 1)

def psnr(a, b):
    mse = float(((a.astype(np.float64) - b.astype(np.float64)) ** 2).mean())
    return 99.0 if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)

# ---------------- GT ----------------
def build_gt(raw, n, names):
    images, anns, aid = [], [], 0
    for i in range(n):
        images.append({"id": i, "width": W, "height": H})
        for o in raw.get(str(2 * i + 1), []):
            x1, y1, x2, y2 = o["x_min"] * W, o["y_min"] * H, o["x_max"] * W, o["y_max"] * H
            w, h = x2 - x1, y2 - y1
            if w <= 0 or h <= 0: continue
            aid += 1
            anns.append({"id": aid, "image_id": i, "category_id": int(o["class_id"]), "bbox": [x1, y1, w, h],
                         "area": w * h, "iscrowd": int(o.get("iscrowd", 0)), "track_id": o.get("track_id")})
    return {"images": images, "annotations": anns, "categories": [{"id": i + 1, "name": n_} for i, n_ in enumerate(names)]}

def size_class(a):
    return "small" if a < 32 ** 2 else ("medium" if a < 96 ** 2 else "large")

# ---------------- 매칭 (B-3·C와 같은 규칙) ----------------
def match_detected(gt_boxes, gt_cls, pred, conf_thr=0.25, iou_thr=0.5):
    det = np.zeros(len(gt_boxes), bool)
    if len(gt_boxes) == 0 or len(pred) == 0: return det
    p = pred[pred[:, 4] >= conf_thr]
    if len(p) == 0: return det
    p = p[np.argsort(-p[:, 4])]
    g = gt_boxes
    gx2, gy2, px2, py2 = g[:, 0] + g[:, 2], g[:, 1] + g[:, 3], p[:, 0] + p[:, 2], p[:, 1] + p[:, 3]
    iw = np.clip(np.minimum(gx2[:, None], px2[None]) - np.maximum(g[:, 0][:, None], p[:, 0][None]), 0, None)
    ih = np.clip(np.minimum(gy2[:, None], py2[None]) - np.maximum(g[:, 1][:, None], p[:, 1][None]), 0, None)
    inter = iw * ih
    iou = inter / np.maximum((g[:, 2] * g[:, 3])[:, None] + (p[:, 2] * p[:, 3])[None] - inter, 1e-9)
    iou = np.where(gt_cls[:, None] == p[None, :, 5], iou, 0.0)
    for j in range(len(p)):
        col = np.where(det, -1.0, iou[:, j]); k = int(np.argmax(col))
        if col[k] >= iou_thr: det[k] = True
    return det

def coco_ap(gt, preds, classes):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    if not preds: return None
    with contextlib.redirect_stdout(io.StringIO()):
        cg = COCO(); cg.dataset = gt; cg.createIndex(); cd = cg.loadRes(preds)
        E = COCOeval(cg, cd, "bbox"); E.params.catIds = classes
        E.evaluate(); E.accumulate(); E.summarize()
    f = lambda x: None if x < 0 else round(100 * x, 2)
    s = E.stats
    return {"AP": f(s[0]), "AP50": f(s[1]), "APs": f(s[3]), "APm": f(s[4]), "APl": f(s[5]), "ARs": f(s[9])}

def cluster_boot(cid, fn, B=1000, seed=0):
    rng = np.random.default_rng(seed); K = int(cid.max()) + 1
    point = fn(np.ones(len(cid))); vals = []
    for _ in range(B):
        cnt = np.bincount(rng.integers(0, K, K), minlength=K)
        v = fn(cnt[cid].astype(float))
        if v is not None and np.isfinite(v): vals.append(v)
    if not vals: return point, np.nan, np.nan
    return point, float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# ---------------- E3: PyAV로 디코딩하면서 매크로블록 QP 지도와 프레임 유형 추출 ----------------
def decode_with_qp(video, keep, w=1280, h=720):
    """반환: {i: (y,u,v)}, {i: qp_map (h/16, w/16)}, {i: 'I'|'P'|'B'}"""
    import av
    c = av.open(video); s = c.streams.video[0]
    s.codec_context.options = {"export_side_data": "venc_params"}
    T = av.sidedata.sidedata.Type
    frames, qps, ptypes = {}, {}, {}
    for i, fr in enumerate(c.decode(s)):
        if i not in keep: continue
        sd = fr.side_data.get(T.VIDEO_ENC_PARAMS)
        if sd is None: raise RuntimeError("QP side data 없음: PyAV/FFmpeg 버전 확인")
        qps[i] = np.asarray(sd.qp_map(), dtype=np.float32)
        ptypes[i] = {1: "I", 2: "P", 3: "B"}.get(int(fr.pict_type), str(fr.pict_type))
        a = fr.to_ndarray(format="yuv420p")            # (h*3/2, w)
        frames[i] = (a[:h].copy(), a[h:h + h // 4].reshape(h // 2, w // 2).copy(),
                     a[h + h // 4:].reshape(h // 2, w // 2).copy())
    c.close()
    return frames, qps, ptypes

def box_mb(b, mbw=80, mbh=45, mb=16):
    x, y, w, h = b
    x0, y0 = int(max(0, x // mb)), int(max(0, y // mb))
    x1, y1 = int(min(mbw, -(-(x + w) // mb))), int(min(mbh, -(-(y + h) // mb)))
    return x0, max(x0 + 1, x1), y0, max(y0 + 1, y1)

def qp_features(q, b):
    """객체 매크로블록 평균·최대 QP, 같은 행 배경(객체 열 제외) 평균 QP, 프레임 평균 QP"""
    x0, x1, y0, y1 = box_mb(b, q.shape[1], q.shape[0])
    obj = q[y0:y1, x0:x1]
    rows = q[y0:y1].copy(); rows[:, x0:x1] = np.nan
    bg = np.nanmean(rows) if np.isfinite(rows).any() else np.nan
    return float(obj.mean()), float(obj.max()), float(bg), float(q.mean())

# ---------------- E4: 레이트 컨트롤 설정별 인코딩, 전체 프레임 디코딩 ----------------
def has_encoder(name):
    r = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
    return any(len(l.split()) > 1 and l.split()[1] == name for l in r.splitlines())

def encode_args(frames, w, h, out, vargs, fps=30):
    """vargs: 코덱·레이트 컨트롤 인자 리스트 (예: ['-c:v','libx264',...])"""
    cmd = ["ffmpeg", "-y", "-v", "error", "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{w}x{h}", "-r", str(fps),
           "-i", "-"] + vargs + ["-an", out]
    p = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    for (y, u, v) in frames:
        p.stdin.write(y.tobytes()); p.stdin.write(u.tobytes()); p.stdin.write(v.tobytes())
    p.stdin.close(); p.wait()
    if p.returncode: raise RuntimeError(f"encode failed: {out}")

def decode_all_with_qp(video, w=1280, h=720):
    """전체 프레임: (yuv 리스트, 프레임 평균 QP 배열(없으면 NaN), 프레임 유형 리스트, 패킷 바이트 배열)"""
    import av
    c = av.open(video); s = c.streams.video[0]
    s.codec_context.options = {"export_side_data": "venc_params"}
    T = av.sidedata.sidedata.Type
    yuv, qp, pt, nb = [], [], [], []
    for pkt in c.demux(s):
        size = pkt.size
        for fr in pkt.decode():
            sd = fr.side_data.get(T.VIDEO_ENC_PARAMS)
            qp.append(float(np.asarray(sd.qp_map()).mean()) if sd is not None else np.nan)
            pt.append({1: "I", 2: "P", 3: "B"}.get(int(fr.pict_type), str(fr.pict_type)))
            nb.append(size)
            a = fr.to_ndarray(format="yuv420p")
            yuv.append((a[:h].copy(), a[h:h + h // 4].reshape(h // 2, w // 2).copy(), a[h + h // 4:].reshape(h // 2, w // 2).copy()))
    c.close()
    return yuv, np.array(qp), pt, np.array(nb)

# ---------------- E6: 색차 처리 ----------------
GRAY_U = np.full((360, 640), 128, np.uint8)

def gray_bgr_from_y(y):
    return yuv_to_bgr((y, GRAY_U, GRAY_U))

def bgr_to_uv(bgr):
    """BGR -> I420의 U, V (yuv_to_bgr과 같은 BT.601 limited range의 역변환)"""
    import cv2
    yuv = cv2.cvtColor(bgr, cv2.COLOR_BGR2YUV_I420)
    h, w = bgr.shape[:2]
    u = yuv[h:h + h // 4].reshape(h // 2, w // 2); v = yuv[h + h // 4:].reshape(h // 2, w // 2)
    return u.copy(), v.copy()

def chroma_pearson(u1, v1, u2, v2):
    a = np.concatenate([u1.ravel(), v1.ravel()]).astype(np.float64)
    b = np.concatenate([u2.ravel(), v2.ravel()]).astype(np.float64)
    if a.std() == 0 or b.std() == 0: return 0.0
    return float(np.corrcoef(a, b)[0, 1])

# ---------------- E7: 색차 업샘플링 (모든 조건에 같은 BT.601 limited 변환) ----------------
def yuv444_to_bgr(y, U, V):
    """y: (H,W) uint8, U,V: (H,W) float/uint8 전해상도 색차 -> BGR uint8. OpenCV I420과 같은 계수"""
    Y = 1.164 * (y.astype(np.float32) - 16.0); u = U.astype(np.float32) - 128.0; v = V.astype(np.float32) - 128.0
    r = Y + 1.596 * v; g = Y - 0.813 * v - 0.391 * u; b = Y + 2.018 * u
    return np.clip(np.dstack([b, g, r]) + 0.5, 0, 255).astype(np.uint8)

def up_nearest(c):
    return np.repeat(np.repeat(c, 2, axis=0), 2, axis=1).astype(np.float32)

def up_bilinear(c):
    import cv2
    h, w = c.shape
    return cv2.resize(c.astype(np.float32), (w * 2, h * 2), interpolation=cv2.INTER_LINEAR)

def _box(x, r):
    import cv2
    return cv2.boxFilter(x, -1, (2 * r + 1, 2 * r + 1), borderType=cv2.BORDER_REFLECT)

def guided_filter(I, p, r, eps):
    """He et al. 가이드 필터. I: 길잡이(휘도), p: 입력(업샘플된 색차), 값 범위 0~1"""
    mI, mp = _box(I, r), _box(p, r)
    vI = _box(I * I, r) - mI * mI; cIp = _box(I * p, r) - mI * mp
    a = cIp / (vI + eps); b = mp - a * mI
    return _box(a, r) * I + _box(b, r)

def up_guided(y, c, r=4, eps=1e-3):
    I = y.astype(np.float32) / 255.0; p = up_bilinear(c) / 255.0
    return np.clip(guided_filter(I, p, r, eps), 0, 1) * 255.0

def up_lm(y, c, r=2, eps=1e-3):
    """저해상도에서 색차 = a·휘도 + b 국소 선형 모델을 맞추고, 원해상도 휘도에 적용 (빠른 가이드 업샘플링, CCLM과 같은 원리)"""
    import cv2
    h, w = c.shape
    Il = cv2.resize(y, (w, h), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0; pl = c.astype(np.float32) / 255.0
    mI, mp = _box(Il, r), _box(pl, r)
    vI = _box(Il * Il, r) - mI * mI; cIp = _box(Il * pl, r) - mI * mp
    a = cIp / (vI + eps); b = mp - a * mI
    a, b = _box(a, r), _box(b, r)
    A = cv2.resize(a, (w * 2, h * 2), interpolation=cv2.INTER_LINEAR); B = cv2.resize(b, (w * 2, h * 2), interpolation=cv2.INTER_LINEAR)
    return np.clip(A * (y.astype(np.float32) / 255.0) + B, 0, 1) * 255.0

# ---------------- 실험 C: 8x8 블록 DCT 대역 개입 ----------------
from scipy.fft import dctn, idctn
def bdct(y):
    return dctn(y.reshape(H // 8, 8, W // 8, 8).transpose(0, 2, 1, 3), axes=(2, 3), norm="ortho")
def ibdct(c):
    return idctn(c, axes=(2, 3), norm="ortho").transpose(0, 2, 1, 3).reshape(H, W)
_UV = np.add.outer(np.arange(8), np.arange(8))
BAND = {"Low": (_UV >= 1) & (_UV <= 2), "Mid": (_UV >= 3) & (_UV <= 6), "High": _UV >= 7}
def band_replace(Dc, Dr, band):
    out = Dc.copy(); m = BAND[band]; out[..., m] = Dr[..., m]; return ibdct(out)
def band_remove(Dc, band):
    out = Dc.copy(); out[..., BAND[band]] = 0.0; return ibdct(out)
def letterbox_params(imgsz, stride=32):
    r = min(imgsz / H, imgsz / W); nw, nh = round(W * r), round(H * r)
    return r, (-(-nw // stride) * stride - nw) / 2, (-(-nh // stride) * stride - nh) / 2, -(-nh // stride) * stride

def to_u8(y):
    return np.clip(np.rint(y), 0, 255).astype(np.uint8)


Writing cd_lib.py


In [3]:
# ===== 설정 =====
TEST_RUN = False       # True: HighwayView 앞 60프레임, 500 kbps만 / False: 전체
RUN_C, RUN_D = True, True
DETECTORS = {"yolo26s": "yolo26s.pt"}      # 인수인계 문서대로 YOLO26s. 필요하면 "yolo26m": "yolo26m.pt" 추가 (D는 YOLO 계열만)
IMGSZS = [640, 1280]   # 640 = A-3과 같은 입력 크기, 1280 = 원본 해상도 (리사이즈 영향 비교)
KBPS = [1000, 500, 250]
STEP = 5; GOP = 60; CONF_MATCH, IOU_MATCH = 0.25, 0.5; N_BOOT = 1000
SEQS = {"HighwayView": "cctv", "CampusView": "cctv",
        "VolleyballGame": "ball", "FloorballGame": "ball", "FloorballTrain": "ball"}
N_LIMIT = None
if TEST_RUN:
    SEQS = {"HighwayView": "cctv"}; N_LIMIT = 60; N_BOOT = 50; KBPS = [500]
# 압축 설정: A-3과 동일 (CCTV형 1-pass CBR, B프레임 없음, GOP 60, veryfast)
X264 = ["-c:v", "libx264", "-preset", "veryfast", "-tune", "zerolatency", "-profile:v", "high", "-pix_fmt", "yuv420p",
        "-g", str(GOP), "-keyint_min", str(GOP), "-sc_threshold", "0"]
def cbr(k): return X264 + ["-b:v", f"{k}k", "-maxrate", f"{k}k", "-bufsize", f"{k}k"]

import os, sys, json, glob, time, shutil
import numpy as np, pandas as pd, torch, cv2
sys.path.insert(0, "/kaggle/working"); import cd_lib as L
assert torch.cuda.is_available(), "GPU 없음 -> Accelerator GPU T4"
OUT = "/kaggle/working/CD_out"; ENC = "/kaggle/working/enc_cd"; os.makedirs(OUT, exist_ok=True); os.makedirs(ENC, exist_ok=True)
REF_ROOT = glob.glob("/kaggle/input/**/uvg_cctv", recursive=True)
assert REF_ROOT, "Input에 uvg_cctv(데이터셋 구축 노트북 Output) 없음"; REF_ROOT = REF_ROOT[0]
BDATA = glob.glob("/kaggle/input/**/bdata", recursive=True); BDATA = BDATA[0] if BDATA else None
GROUP_CLS = {"cctv": L.CCTV_CLASSES, "ball": L.BALL_CLASSES}
print("uvg_cctv:", REF_ROOT, "| bdata:", BDATA)


uvg_cctv: /kaggle/input/notebooks/naeunku/uvg-build-dataset/uvg_cctv | bdata: /kaggle/input/notebooks/naeunku/uvg-b-data-prep/bdata


In [4]:
# ===== 탐지기 =====
from ultralytics import YOLO
MODELS = {k: YOLO(w) for k, w in DETECTORS.items()}
NAMES = [list(MODELS.values())[0].names[i] for i in range(80)]
def to_bgr(f, sz):
    return L.yuv_to_bgr(f if sz == 1280 else L.resize_yuv(f, 640, 360))
def predict(m, imgs, sz):
    sc = 1.0 if sz == 1280 else 2.0; out = []
    for s in range(0, len(imgs), 16):
        for r in m.predict(imgs[s:s + 16], imgsz=sz, conf=0.001, max_det=300, device=0, verbose=False, batch=len(imgs[s:s + 16])):
            b = r.boxes; xy = b.xyxy.cpu().numpy() * sc
            out.append(np.column_stack([xy[:, 0], xy[:, 1], xy[:, 2] - xy[:, 0], xy[:, 3] - xy[:, 1], b.conf.cpu().numpy(), b.cls.cpu().numpy() + 1]))
    return out
print(list(MODELS))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
['yolo26s']


In [5]:
# ===== 데이터 준비: 기준 영상, GT, 압축 영상 (bdata에 A-3 영상이 있으면 재사용, 없으면 같은 명령으로 압축) =====
SEQ = {}
for si, (s, grp) in enumerate(SEQS.items()):
    n = json.load(open(f"{REF_ROOT}/{s}/done.json"))["ref_frames"]
    if N_LIMIT: n = min(n, N_LIMIT)
    keep = list(range(0, n, STEP))
    gt = L.build_gt(json.load(open(f"{REF_ROOT}/{s}/ann_raw.json")), n, NAMES)
    anns = {i: [a for a in gt["annotations"] if a["image_id"] == i and a["category_id"] in GROUP_CLS[grp]] for i in keep}
    vids = {}
    for k in KBPS:
        p = f"{BDATA}/{s}/cctv_{k}k.mp4" if BDATA else None
        if not (p and os.path.exists(p)) or N_LIMIT:
            ref_all, _ = L.read_yuv(f"{REF_ROOT}/{s}/ref_720p30.mkv", 1280, 720)
            p = f"{ENC}/{s}_{k}.mp4"; L.encode_args(ref_all[:n], 1280, 720, p, cbr(k)); del ref_all
        vids[k] = p
    SEQ[s] = {"si": si, "group": grp, "n": n, "keep": keep, "gt": gt, "anns": anns, "vids": vids,
              "gb": {i: np.array([a["bbox"] for a in anns[i]], float).reshape(-1, 4) for i in keep},
              "gc": {i: np.array([a["category_id"] for a in anns[i]]) for i in keep}}
    print(s, "프레임", n, "평가", len(keep), "| 압축 영상:", {k: L.kbps_of(v) for k, v in vids.items()})
OBJ = pd.DataFrame([{"seq": s, "group": v["group"], "frame": i, "ann_id": a["id"], "area": a["area"], "size": L.size_class(a["area"]),
                     "track": f"{s}:{a['track_id'] if a['track_id'] is not None else 'a' + str(a['id'])}"}
                    for s, v in SEQ.items() for i in v["keep"] for a in v["anns"][i]])


HighwayView 프레임 300 평가 60 | 압축 영상: {1000: 940.5, 500: 463.6, 250: 225.2}
CampusView 프레임 300 평가 60 | 압축 영상: {1000: 986.5, 500: 483.2, 250: 233.9}
VolleyballGame 프레임 300 평가 60 | 압축 영상: {1000: 968.9, 500: 485.2, 250: 243.3}
FloorballGame 프레임 210 평가 42 | 압축 영상: {1000: 1018.0, 500: 509.8, 250: 257.4}
FloorballTrain 프레임 300 평가 60 | 압축 영상: {1000: 1017.5, 500: 508.9, 250: 256.2}


In [6]:
# ===== 실험 C: 주파수 대역 교체 (Oracle Test) =====
# 조건 (인수인계 문서) Baseline=압축 그대로 / Low·Mid·High = 해당 대역만 원본 계수로 교체 / Reference=무손실 원본
# 추가 1개: High_remove = 압축 영상의 고주파를 0으로 (교체 효과가 "정보 복원"인지 "아티팩트 제거"인지 구분)
# 변환: 8x8 블록 DCT (휘도 Y), DC 고정. 색차는 압축본 그대로(Reference만 원본). 모든 조건 같은 색변환
C_CONDS = ["Baseline", "Low", "Mid", "High", "High_remove", "Reference"]
CFLAG, CPRED = {}, {}
if RUN_C:
    t0 = time.time()
    for s, v in SEQ.items():
        keep = v["keep"]; R, _ = L.read_yuv(f"{REF_ROOT}/{s}/ref_720p30.mkv", 1280, 720, keep=set(keep))
        DR = {i: L.bdct(R[i][0].astype(np.float64)) for i in keep}
        for k in KBPS:
            C_, _ = L.read_yuv(v["vids"][k], 1280, 720, keep=set(keep))
            frames = {c: {} for c in C_CONDS}
            for i in keep:
                yc = C_[i][0].astype(np.float64); Dc = L.bdct(yc); uv = (C_[i][1], C_[i][2])
                frames["Baseline"][i] = (L.to_u8(L.ibdct(Dc)), *uv)
                for b in ["Low", "Mid", "High"]: frames[b][i] = (L.to_u8(L.band_replace(Dc, DR[i], b)), *uv)
                frames["High_remove"][i] = (L.to_u8(L.band_remove(Dc, "High")), *uv)
                frames["Reference"][i] = R[i]
            for cond in C_CONDS:
                for sz in IMGSZS:
                    imgs = [to_bgr(frames[cond][i], sz) for i in keep]
                    for dk, m in MODELS.items():
                        preds = predict(m, imgs, sz); key = (dk, k, cond, sz)
                        CPRED.setdefault(key, []).extend((v["si"] * 100000 + i, p) for i, p in zip(keep, preds))
                        CFLAG.setdefault(key, {})[s] = np.concatenate([L.match_detected(v["gb"][i], v["gc"][i], p, CONF_MATCH, IOU_MATCH)
                                                                       for i, p in zip(keep, preds)] or [np.zeros(0, bool)])
            print(f"C {s:15s} {k} kbps 완료 ({time.time()-t0:.0f}s)", flush=True)


C HighwayView     1000 kbps 완료 (38s)
C HighwayView     500 kbps 완료 (58s)
C HighwayView     250 kbps 완료 (78s)
C CampusView      1000 kbps 완료 (103s)
C CampusView      500 kbps 완료 (124s)
C CampusView      250 kbps 완료 (144s)
C VolleyballGame  1000 kbps 완료 (170s)
C VolleyballGame  500 kbps 완료 (191s)
C VolleyballGame  250 kbps 완료 (213s)
C FloorballGame   1000 kbps 완료 (231s)
C FloorballGame   500 kbps 완료 (247s)
C FloorballGame   250 kbps 완료 (261s)
C FloorballTrain  1000 kbps 완료 (288s)
C FloorballTrain  500 kbps 완료 (310s)
C FloorballTrain  250 kbps 완료 (331s)


In [7]:
# ===== 실험 C 분석: APs·APm·APl, 원본 대비 회복량, 대역별 회복 기여도 =====
def merged_gt(grp):
    g = {"images": [], "annotations": [], "categories": list(SEQ.values())[0]["gt"]["categories"]}; aid = 0
    for s, v in SEQ.items():
        if v["group"] != grp: continue
        keep = set(v["keep"])
        g["images"] += [{**im, "id": v["si"] * 100000 + im["id"]} for im in v["gt"]["images"] if im["id"] in keep]
        for a in v["gt"]["annotations"]:
            if a["image_id"] in keep and a["category_id"] in GROUP_CLS[grp]:
                aid += 1; g["annotations"].append({**a, "id": aid, "image_id": v["si"] * 100000 + a["image_id"]})
    return g
if RUN_C:
    rows = []
    for grp in set(v["group"] for v in SEQ.values()):
        g = merged_gt(grp); uids = {im["id"] for im in g["images"]}; cls = GROUP_CLS[grp]
        for (dk, k, cond, sz), lst in CPRED.items():
            preds = [{"image_id": int(u), "category_id": int(r[5]), "bbox": [float(x) for x in r[:4]], "score": float(r[4])}
                     for u, arr in lst if u in uids for r in arr if int(r[5]) in cls]
            m = L.coco_ap(g, preds, cls)
            if m: rows.append({"det": dk, "group": grp, "kbps": k, "cond": cond, "imgsz": sz, **m})
    cap = pd.DataFrame(rows)
    for met in ["APs", "APm", "APl", "AP"]:   # 회복량 (%p)과 회복 비율 (Baseline -> Reference 격차 대비)
        b = cap[cap.cond == "Baseline"].set_index(["det", "group", "kbps", "imgsz"])[met]
        t = cap[cap.cond == "Reference"].set_index(["det", "group", "kbps", "imgsz"])[met]
        key = list(zip(cap.det, cap.group, cap.kbps, cap.imgsz))
        cap[f"{met}_gain"] = [None if (x is None or b.get(kk) is None) else round(x - b[kk], 2) for x, kk in zip(cap[met], key)]
        cap[f"{met}_recovery_%"] = [None if (x is None or b.get(kk) is None or t.get(kk) is None or t[kk] == b[kk]) else round((x - b[kk]) / (t[kk] - b[kk]) * 100, 1)
                                    for x, kk in zip(cap[met], key)]
    cap.to_csv(f"{OUT}/C_ap.csv", index=False)
    # 객체 단위: 작은 객체 회복률 (Reference에서 찾았지만 Baseline에서 놓친 객체 중 다시 찾은 비율), track 단위 bootstrap
    objc = OBJ.copy()
    for key in CFLAG: objc["|".join(map(str, key))] = np.concatenate([CFLAG[key][s] for s in SEQ])
    objc.to_csv(f"{OUT}/C_object.csv", index=False)
    rec = []
    for grp in objc.group.unique():
        for size in ["small", "medium", "large"]:
            d = objc[(objc.group == grp) & (objc["size"] == size)].reset_index(drop=True)
            if len(d) < 30: continue
            tr = d.track.astype("category").cat.codes.to_numpy()
            for dk in MODELS:
                for k in KBPS:
                    for sz in IMGSZS:
                        c = lambda cond: d[f"{dk}|{k}|{cond}|{sz}"].to_numpy(float)
                        ref, base = c("Reference"), c("Baseline"); m = (ref == 1) & (base == 0)
                        for cond in ["Low", "Mid", "High", "High_remove"]:
                            x = c(cond)
                            fn = lambda w, x=x: (w * m * x).sum() / max((w * m).sum(), 1e-9)
                            p, lo, hi = L.cluster_boot(tr, fn, B=N_BOOT)
                            rec.append({"group": grp, "size": size, "det": dk, "kbps": k, "imgsz": sz, "cond": cond, "n_missed": int(m.sum()),
                                        "recovery": p, "lo": lo, "hi": hi})
    rec = pd.DataFrame(rec); rec.to_csv(f"{OUT}/C_object_recovery.csv", index=False)
    pd.set_option("display.width", 250)
    print(cap.pivot_table(index=["group", "kbps", "cond"], columns=["det", "imgsz"], values="APs").reindex(C_CONDS, level=2).round(2).to_string())
    print(rec[rec["size"] == "small"].pivot_table(index=["group", "det", "kbps", "imgsz"], columns="cond", values="recovery").mul(100).round(1).to_string())
    # 시퀀스별 (연속 프레임 상관 때문에 시퀀스 단위로도 보고)
    per = []
    for s in SEQ:
        d = objc[(objc.seq == s) & (objc["size"] == "small")]
        if len(d) < 10: continue
        for dk in MODELS:
            for k in KBPS:
                for sz in IMGSZS:
                    m = (d[f"{dk}|{k}|Reference|{sz}"] & ~d[f"{dk}|{k}|Baseline|{sz}"])
                    per.append({"seq": s, "det": dk, "kbps": k, "imgsz": sz, "n_missed": int(m.sum()),
                                **{f"rec_{c}": round(d.loc[m, f"{dk}|{k}|{c}|{sz}"].mean() * 100, 1) if m.sum() else None for c in ["Low", "Mid", "High", "High_remove"]}})
    pd.DataFrame(per).to_csv(f"{OUT}/C_per_sequence.csv", index=False)


det                    yolo26s       
imgsz                     640    1280
group kbps cond                      
ball  250  Baseline       5.41   6.59
           Low            7.14   9.74
           Mid            6.86  13.55
           High           5.63   9.24
           High_remove    5.44   6.86
           Reference     11.81  26.10
      500  Baseline       9.33  15.10
           Low            9.38  16.18
           Mid            9.53  18.81
           High           9.26  16.58
           High_remove    9.22  15.10
           Reference     11.81  26.10
      1000 Baseline      10.30  19.36
           Low           10.43  19.99
           Mid           10.56  22.24
           High          10.25  20.54
           High_remove   10.22  18.16
           Reference     11.81  26.10
cctv  250  Baseline       9.06  11.45
           Low           14.04  14.47
           Mid           12.82  25.93
           High           9.46  16.27
           High_remove    8.97  10.51
           R

In [8]:
# ===== 실험 D: 탐지기 내부 Feature Drift =====
# 같은 프레임의 [원본, 1000k, 500k, 250k]를 한 배치로 넣고, Backbone(stride 4·8·16·32 마지막 층)과 Neck 출력(P3·P4·P5 = Detect 입력)을 비교
# 지표: 상대 오차 ||f - f_ref||^2 / ||f_ref||^2, 코사인 유사도, 채널 정규화 MSE(원본 통계로 표준화), 작은 객체 영역 버전
# 실제 층 번호는 YOLO26s 구현에서 자동으로 찾음 (첫 Upsample 이전 = Backbone)
DROWS, DMISS = [], []
if RUN_D:
    dk, m = next((k, v) for k, v in MODELS.items())
    # C에서 이미 predict를 돌렸으면 실제 추론은 predictor 안의 모델로 일어남 -> 그 모델에 hook을 걸어야 함
    if getattr(m, "predictor", None) is None:
        _ = m.predict(np.zeros((720, 1280, 3), np.uint8), imgsz=640, device=0, verbose=False)
    run_model = m.predictor.model.model if hasattr(m.predictor.model, "model") else m.model
    seq_mod = run_model.model; FE = {}; SEL = {}; KEEP = None   # KEEP: 저장할 층 번호 (처음 한 번만 전체 저장해 층을 고른 뒤 선택 층만 저장 -> GPU 메모리 절약)
    def _hook(mod, inp, out):
        if torch.is_tensor(out) and (KEEP is None or mod.i in KEEP): FE[mod.i] = out.detach()
    hooks = [mod.register_forward_hook(_hook) for mod in seq_mod]
    head = seq_mod[-1]; hooks.append(head.register_forward_pre_hook(lambda mod, inp: FE.__setitem__("neck", [t.detach() for t in inp[0]])))
    up_idx = min([mod.i for mod in seq_mod if "Upsample" in type(mod).__name__] or [len(seq_mod) // 2])   # 첫 Upsample = Neck 시작
    def select_layers(hpad):
        sel = {}
        for i, t in FE.items():
            if i == "neck" or i >= up_idx or t.dim() != 4: continue
            st = int(round(hpad / t.shape[2]))
            if st in (4, 8, 16, 32): sel[f"backbone_s{st}"] = max(sel.get(f"backbone_s{st}", -1), i)
        return sel
    t0 = time.time()
    for s, v in SEQ.items():
        keep = v["keep"]; R, _ = L.read_yuv(f"{REF_ROOT}/{s}/ref_720p30.mkv", 1280, 720, keep=set(keep))
        C_ = {k: L.read_yuv(v["vids"][k], 1280, 720, keep=set(keep))[0] for k in KBPS}
        for sz in IMGSZS:
            r_, pw, ph, hpad = L.letterbox_params(sz)
            for i in keep:
                imgs = [to_bgr(R[i], sz)] + [to_bgr(C_[k][i], sz) for k in KBPS]
                if KEEP is None:   # 640 입력 1장으로 층 고르기 (층 번호는 입력 크기와 무관)
                    FE.clear(); predict(m, [to_bgr(R[i], 640)], 640)
                    _, _, _, hp640 = L.letterbox_params(640); SEL0 = select_layers(hp640); KEEP = set(SEL0.values())
                    assert "neck" in FE and SEL0, f"hook 미작동: FE 키 {list(FE)[:10]} -> 로그 전달"
                    print("선택 층:", SEL0, "| Detect 입력 수:", len(FE["neck"])); torch.cuda.empty_cache()
                FE.clear(); preds = predict(m, imgs, sz)
                feats = {name: FE[idx] for name, idx in SEL0.items()}
                for lv, t in zip(["neck_P3", "neck_P4", "neck_P5"], FE["neck"]): feats[lv] = t
                small = [a["bbox"] for a in v["anns"][i] if L.size_class(a["area"]) == "small"]
                for name, f in feats.items():
                    f = f.float(); ref = f[0]; st = hpad / f.shape[2]
                    mu = ref.mean(dim=(1, 2), keepdim=True); sd = ref.std(dim=(1, 2), keepdim=True) + 1e-6
                    mask = torch.zeros(f.shape[2:], dtype=torch.bool, device=f.device)
                    for (x, y, w, h) in small:
                        x0, y0 = int((x * r_ + pw) // st), int((y * r_ + ph) // st)
                        x1 = max(x0 + 1, int(-(-((x + w) * r_ + pw) // st))); y1 = max(y0 + 1, int(-(-((y + h) * r_ + ph) // st)))
                        mask[y0:y1, x0:x1] = True
                    cos = torch.nn.functional.cosine_similarity(f, ref[None], dim=1)
                    for j, k in enumerate(KBPS, start=1):
                        dd = (f[j] - ref) ** 2
                        row = {"seq": s, "group": v["group"], "frame": i, "imgsz": sz, "kbps": k, "layer": name, "stride": round(st),
                               "rel_err": float(dd.sum() / (ref ** 2).sum()), "cos": float(cos[j].mean()),
                               "norm_mse": float((((f[j] - mu) / sd - (ref - mu) / sd) ** 2).mean())}
                        if mask.any():
                            row["rel_err_small"] = float(dd[:, mask].sum() / (ref[:, mask] ** 2).sum()); row["cos_small"] = float(cos[j][mask].mean())
                        DROWS.append(row)
                del feats; FE.clear()
                # 같은 배치의 탐지 결과로 프레임별 작은 객체 미탐률 (원본에서 찾은 작은 객체 중 압축에서 놓친 비율)
                sm = np.array([L.size_class(a["area"]) == "small" for a in v["anns"][i]], bool)
                f_ref = L.match_detected(v["gb"][i], v["gc"][i], preds[0], CONF_MATCH, IOU_MATCH)
                for j, k in enumerate(KBPS, start=1):
                    f_c = L.match_detected(v["gb"][i], v["gc"][i], preds[j], CONF_MATCH, IOU_MATCH); base = f_ref & sm
                    DMISS.append({"seq": s, "frame": i, "imgsz": sz, "kbps": k, "n_ref_small": int(base.sum()),
                                  "miss_small": float((base & ~f_c).sum() / base.sum()) if base.sum() else np.nan})
        print(f"D {s:15s} 완료 ({time.time()-t0:.0f}s)", flush=True)
    for h in hooks: h.remove()


선택 층: {'backbone_s4': 2, 'backbone_s8': 4, 'backbone_s16': 6, 'backbone_s32': 10} | Detect 입력 수: 3
D HighwayView     완료 (19s)
D CampusView      완료 (39s)
D VolleyballGame  완료 (59s)
D FloorballGame   완료 (72s)
D FloorballTrain  완료 (91s)


In [9]:
# ===== 실험 D 분석: 단계별 drift, 압축 강도별 변화, 주입 위치 후보 =====
if RUN_D:
    dfe = pd.DataFrame(DROWS); dfe.to_csv(f"{OUT}/D_feature_drift.csv", index=False)
    dms = pd.DataFrame(DMISS); dms.to_csv(f"{OUT}/D_frame_miss.csv", index=False)
    order = [f"backbone_s{s}" for s in (4, 8, 16, 32)] + ["neck_P3", "neck_P4", "neck_P5"]
    summ = dfe.groupby(["group", "imgsz", "kbps", "layer"])[["rel_err", "cos", "norm_mse", "rel_err_small", "cos_small"]].mean().reset_index()
    summ["layer"] = pd.Categorical(summ.layer, [o for o in order if o in summ.layer.unique()], ordered=True)
    summ = summ.sort_values(["group", "imgsz", "kbps", "layer"]); summ.to_csv(f"{OUT}/D_summary.csv", index=False)
    pd.set_option("display.width", 250); print(summ.round(3).to_string(index=False))
    # 주입 위치 후보 근거: 프레임별 작은 객체 영역 drift가 프레임별 작은 객체 미탐률과 얼마나 같이 움직이나 (시퀀스·비트레이트 안에서 순위 상관 평균)
    from scipy.stats import spearmanr
    mg = dfe.merge(dms, on=["seq", "frame", "imgsz", "kbps"])
    cand = []
    for (grp, sz, layer), g in mg.groupby(["group", "imgsz", "layer"]):
        rhos = [spearmanr(gg.rel_err_small, gg.miss_small).correlation for _, gg in g.groupby(["seq", "kbps"])
                if gg.rel_err_small.notna().sum() > 5 and gg.miss_small.notna().sum() > 5 and gg.miss_small.nunique() > 1]
        cand.append({"group": grp, "imgsz": sz, "layer": layer, "n_groups": len(rhos), "spearman_drift_small_vs_miss": np.nanmean(rhos) if rhos else np.nan})
    cand = pd.DataFrame(cand); cand.to_csv(f"{OUT}/D_injection_candidates.csv", index=False)
    print("\n주입 위치 후보 (프레임별 작은 객체 영역 drift와 미탐률의 순위 상관, 클수록 탐지 실패와 강하게 연결)")
    print(cand.sort_values(["group", "imgsz", "spearman_drift_small_vs_miss"], ascending=[True, True, False]).round(3).to_string(index=False))
    print("\n※ drift가 가장 큰 층이 최적 주입 위치라는 뜻은 아님. 후보 선정 후 소규모 성능 검증 필요")


group  imgsz  kbps        layer  rel_err   cos  norm_mse  rel_err_small  cos_small
 ball    640   250  backbone_s4    0.458 0.744     0.739          0.601      0.625
 ball    640   250  backbone_s8    0.416 0.793     0.883          0.506      0.705
 ball    640   250 backbone_s16    0.405 0.800     0.916          0.452      0.748
 ball    640   250 backbone_s32    0.439 0.775     2.140          0.412      0.773
 ball    640   250      neck_P3    0.168 0.918     0.854          0.257      0.862
 ball    640   250      neck_P4    0.313 0.845     0.784          0.391      0.778
 ball    640   250      neck_P5    0.421 0.803     0.840          0.496      0.751
 ball    640   500  backbone_s4    0.273 0.840     0.457          0.328      0.811
 ball    640   500  backbone_s8    0.225 0.887     0.482          0.268      0.855
 ball    640   500 backbone_s16    0.206 0.899     0.472          0.253      0.864
 ball    640   500 backbone_s32    0.204 0.898     0.877          0.154      0.919
 bal

/tmp/ipykernel_22/4142472860.py:17: RuntimeWarning: Mean of empty slice
  cand.append({"group": grp, "imgsz": sz, "layer": layer, "n_groups": len(rhos), "spearman_drift_small_vs_miss": np.nanmean(rhos) if rhos else np.nan})
/tmp/ipykernel_22/4142472860.py:17: RuntimeWarning: Mean of empty slice
  cand.append({"group": grp, "imgsz": sz, "layer": layer, "n_groups": len(rhos), "spearman_drift_small_vs_miss": np.nanmean(rhos) if rhos else np.nan})
/tmp/ipykernel_22/4142472860.py:17: RuntimeWarning: Mean of empty slice
  cand.append({"group": grp, "imgsz": sz, "layer": layer, "n_groups": len(rhos), "spearman_drift_small_vs_miss": np.nanmean(rhos) if rhos else np.nan})
/tmp/ipykernel_22/4142472860.py:17: RuntimeWarning: Mean of empty slice
  cand.append({"group": grp, "imgsz": sz, "layer": layer, "n_groups": len(rhos), "spearman_drift_small_vs_miss": np.nanmean(rhos) if rhos else np.nan})
/tmp/ipykernel_22/4142472860.py:17: RuntimeWarning: Mean of empty slice
  cand.append({"group": grp, "im

In [10]:
# ===== 그림 (영어 라벨) =====
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
if RUN_C:
    for grp in cap.group.unique():
        z = cap[(cap.group == grp)]
        fig, axs = plt.subplots(1, len(IMGSZS), figsize=(5.5 * len(IMGSZS), 3.6), squeeze=False)
        for ax, sz in zip(axs[0], IMGSZS):
            q = z[(z.imgsz == sz) & (z.det == list(MODELS)[0])].pivot_table(index="kbps", columns="cond", values="APs")[C_CONDS]
            q.plot.bar(ax=ax, rot=0); ax.set_ylabel("APs"); ax.set_title(f"C: {grp}, imgsz {sz}"); ax.grid(alpha=.3, axis="y"); ax.legend(fontsize=7)
        fig.tight_layout(); fig.savefig(f"{OUT}/C_fig_band_APs_{grp}.png", dpi=130); plt.close(fig)
if RUN_D:
    for grp in summ.group.unique():
        fig, axs = plt.subplots(1, len(IMGSZS), figsize=(5.5 * len(IMGSZS), 3.6), squeeze=False)
        for ax, sz in zip(axs[0], IMGSZS):
            for k in KBPS:
                z = summ[(summ.group == grp) & (summ.imgsz == sz) & (summ.kbps == k)]
                ax.plot(z.layer.astype(str), z.rel_err, "o-", label=f"{k} kbps")
            ax.set_title(f"D: {grp}, imgsz {sz}"); ax.set_ylabel("relative feature error"); ax.tick_params(axis="x", rotation=40); ax.grid(alpha=.3); ax.legend(fontsize=7)
        fig.tight_layout(); fig.savefig(f"{OUT}/D_fig_drift_{grp}.png", dpi=130); plt.close(fig)
shutil.rmtree(ENC, ignore_errors=True)
print(sorted(os.listdir(OUT)))


['C_ap.csv', 'C_fig_band_APs_ball.png', 'C_fig_band_APs_cctv.png', 'C_object.csv', 'C_object_recovery.csv', 'C_per_sequence.csv', 'D_feature_drift.csv', 'D_fig_drift_ball.png', 'D_fig_drift_cctv.png', 'D_frame_miss.csv', 'D_injection_candidates.csv', 'D_summary.csv']
